# 6.5 Slanted and Blazed In-Coupler Gratings: Unidirectional Coupling and Conformal Subpixel Meshing

**Level:** Advanced · **Time:** ~45 s (`--quick`, 2 cores) / ~3 min (`--full`)

In **Tutorial 6.4**, we saw that a symmetric binary surface-relief grating splits a normally
incident display beam equally between the $+1$ and $-1$ diffraction orders ($T_{+1} = T_{-1}$),
wasting at least half of the micro-display's optical power in the wrong direction. Practical
augmented-reality (AR) waveguide combiners overcome this fundamental symmetry limit by breaking
mirror symmetry $x \to -x$ inside the grating unit cell—either with **sawtooth blazed** grooves
or with **slanted high-index ridges** tilted at an angle $\alpha \in [20^\circ, 35^\circ]$
(**Levola, 2006**; **Levola & Laakkonen, 2007**; **Wang & Ma, 2023**, `arXiv:2307.01600`).

Simulating tilted dielectric sidewalls on a Cartesian Yee grid introduces a classic numerical
challenge: naive **staircase** discretization replaces smooth slanted interfaces with artificial
right-angled steps that scatter spurious phase errors and slow convergence to $\mathcal{O}(\Delta x)$.
In this tutorial we combine `wakis.floquet` (`FloquetCell`, `AngleSweep`, `FloquetPort`) with
**Kottke/Farjadpour anisotropic subpixel permittivity smoothing** and cross-check every
diffraction order against a multi-layer Fourier Modal Method (RCWA) solver.

### Learning goals
1. Derive the **Bragg slant condition** $\alpha_{\text{Bragg}} = \frac{1}{2}(|\theta_{-1}| - \theta_{\text{in}})$ and explain why tilting the grating grooves suppresses the parasitic $+1$ order and concentrates $> 65\text{--}80\%$ of transmitted power into the guided $-1$ order (**Wang & Ma, 2023**, Table 1; **Levola & Laakkonen, 2007**).
2. Map first-order guided efficiency $T_{-1}$ and $\pm 1$ selectivity $T_{-1}/T_{+1}$ across **slant angle $\alpha$**, **groove depth $d$**, and **fill factor $f$** at normal and oblique incidence, comparing **Binary**, **Sawtooth Blazed**, and **Slanted** profiles.
3. Formulate and visualize **conformal subpixel permittivity averaging** vs **staircase** discretization on identical Yee meshes, and demonstrate second-order convergence against exact multi-layer RCWA.
4. Perform constant-angle broadband **Floquet sweeps** (`AngleSweep.run()` + `AngleSweepResult.resample()`) for both TE and TM polarizations.

---

## Physics & Step-by-Step Derivations

### 1. Mirror symmetry breaking and the Bragg slant condition
For a 1D grating of period $\Lambda$ along $x$ separating air ($n_{\text{inc}} = 1.0$) from a
waveguide substrate ($n_s = 1.74$, N-LAF2 glass in **Wang & Ma, 2023**), the $m\text{th}$
transmitted order propagates at angle $\theta_m$ inside the glass given by the grating equation:
$$n_s \sin\theta_m = n_{\text{inc}}\sin\theta_{\text{in}} + m\frac{\lambda_0}{\Lambda}, \qquad m \in \mathbb{Z}. \tag{1}$$
If the permittivity profile satisfies inversion symmetry $\varepsilon(-x, z) = \varepsilon(x, z)$
(as in any binary rectangular grating), then at normal incidence ($\theta_{\text{in}} = 0^\circ$)
Maxwell's equations are invariant under $x \to -x$, forcing exact order degeneracy:
$$T_{+1}(\theta_{\text{in}}=0^\circ) \equiv T_{-1}(\theta_{\text{in}}=0^\circ) \le 0.50. \tag{2}$$

Tilting the dielectric ridges by a **slant angle $\alpha$** from the $z$-axis (so the ridge boundary
follows $x(z) = x_0 + z\tan\alpha$ for $z \in [0, d]$) aligns the high-index waveguide channels
with the desired diffracted wavevector $\mathbf{k}_{-1}$. Geometrically, the groove sidewall acts as
an internal Bragg mirror when its surface normal $\hat{\mathbf{n}} = (\cos\alpha, 0, -\sin\alpha)$
bisects the incident wavevector in air ($\theta_{\text{in}}$) and the target $m=-1$ wavevector in
the substrate ($|\theta_{-1}|$):
$$\alpha_{\text{Bragg}}(\theta_{\text{in}}) = \frac{1}{2}\left(|\theta_{-1}| - \theta_{\text{in}}\right)
= \frac{1}{2}\left[\arcsin\!\left(\frac{\lambda_0/\Lambda - \sin\theta_{\text{in}}}{n_s}\right) - \theta_{\text{in}}\right]. \tag{3}$$
At $\lambda_0 = 525\;\text{nm}$, $\Lambda = 400\;\text{nm}$, and $n_s = 1.74$ (**Wang & Ma, 2023**),
$|\theta_{-1}(0^\circ)| = \arcsin(1.3125 / 1.74) = 49.0^\circ$, yielding $\alpha_{\text{Bragg}} = 24.5^\circ$—remarkably
close to the full-wave optimized slant angle $\alpha = 25^\circ$ reported in Table 1 of **Wang & Ma (2023)**!

### 2. Modal phase accumulation: groove depth $d$ and fill factor $f$
Inside a slanted grating of depth $d$ and duty cycle (fill factor) $f \in (0, 1)$, the incident wave
excites two dominant Bloch eigenmodes propagating along the tilted grooves ($\ell = d/\cos\alpha$) with
effective indices $n_{\text{eff},1}(f)$ and $n_{\text{eff},2}(f)$ (**Levola, 2006**). Constructive
interference into the $-1$ order occurs when the differential phase accumulation equals $\pi$:
$$\Delta\phi = \frac{2\pi}{\lambda_0}\left|n_{\text{eff},1}(f) - n_{\text{eff},2}(f)\right|\frac{d}{\cos\alpha} \approx \pi
\quad\Longrightarrow\quad d_{\text{opt}} \approx \frac{\lambda_0 \cos\alpha}{2\,\Delta n_{\text{eff}}(f)}. \tag{4}$$
High-index $\text{TiO}_2$-coated or $\text{TiO}_2$/resin ridges ($n_{\text{ridge}} \approx 2.35\text{--}2.98$)
maximize $\Delta n_{\text{eff}}$, achieving peak first-order coupling at moderate depths $d \in [300, 415]\;\text{nm}$
and fill factors $f \in [0.50, 0.66]$ (**Wang & Ma, 2023**, Table 1: $\alpha = 25^\circ, f = 66\%, d = 415\;\text{nm}$).

### 3. Conformal subpixel permittivity smoothing vs staircase meshing
When a dielectric interface with unit normal $\hat{\mathbf{n}} = (n_x, 0, n_z) = (\cos\alpha, 0, -\sin\alpha)$
cuts through a Cartesian Yee cell, **Farjadpour et al. (2006)** and **Kottke et al. (2008)** proved that
second-order accuracy requires averaging $\varepsilon$ for electric field components **parallel** to the
interface ($\mathbf{P}_\parallel = \mathbf{I} - \hat{\mathbf{n}}\hat{\mathbf{n}}^T$) and $\varepsilon^{-1}$ for
electric displacement components **normal** to the interface ($\mathbf{P}_\perp = \hat{\mathbf{n}}\hat{\mathbf{n}}^T$):
$$\tilde{\boldsymbol{\varepsilon}}^{-1} = \mathbf{P}_\parallel\,\langle\varepsilon\rangle^{-1} + \mathbf{P}_\perp\,\langle\varepsilon^{-1}\rangle. \tag{5}$$
Projecting onto the Cartesian Yee edges $(E_x, E_y, E_z)$ gives the effective diagonal inverse permittivities:
$$\left(\tilde{\varepsilon}^{-1}\right)_{xx} = \frac{\sin^2\alpha}{\langle\varepsilon\rangle} + \cos^2\alpha\,\langle\varepsilon^{-1}\rangle, \quad
\left(\tilde{\varepsilon}^{-1}\right)_{zz} = \frac{\cos^2\alpha}{\langle\varepsilon\rangle} + \sin^2\alpha\,\langle\varepsilon^{-1}\rangle, \quad
\left(\tilde{\varepsilon}^{-1}\right)_{yy} = \frac{1}{\langle\varepsilon\rangle}, \tag{6}$$
where $\langle\varepsilon\rangle$ and $\langle\varepsilon^{-1}\rangle$ are evaluated by $16\times 16$ subcell quadrature over each staggered Yee dual cell.

In [ ]:
from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
import shlex
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Polygon, Rectangle
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import epsilon_0

TUTORIAL_DIR = Path(__file__).resolve().parent
if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))

import wakis
from _helpers import (
    DEFAULT_DEPTH_M,
    N_RIDGE,
    N_SUBSTRATE,
    PERIOD_INCOUPLER_M,
    WANG_DEPTH_M,
    WANG_FILL_FACTOR,
    WANG_SLANT_DEG,
    WL_DESIGN_M,
    bragg_slant_angle_deg,
    build_slanted_floquet_cell,
    run_task_0_slant_and_profiles,
    run_task_1_depth_and_fill_sweeps,
    run_task_2_oblique_and_broadband,
    run_task_3_staircase_vs_conformal,
)

# High-contrast paper figure style
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.size": 9.5,
        "axes.titlesize": 10.5,
        "axes.labelsize": 9.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5,
        "axes.grid": True,
        "grid.alpha": 0.28,
        "grid.linestyle": "--",
        "lines.linewidth": 1.8,
    }
)

PALETTE = {
    "B": "#0072B2",
    "G": "#009E73",
    "R": "#D55E00",
    "gold": "#E69F00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "ink": "#1A1A1A",
    "grey": "#666666",
}

TASKS = {
    0: (
        "slant_angle_and_profile_comparison",
        "Slant angle sweep alpha in [0, 40] deg + Binary vs Blazed vs Slanted order comparison",
    ),
    1: (
        "depth_and_fill_factor_sweeps",
        "Groove depth d and fill factor f sweeps + 2D RCWA design map (Wang & Ma Table 1)",
    ),
    2: (
        "oblique_incidence_and_broadband_sweep",
        "Oblique FOV sweeps (Binary, Blazed, Slanted; TE & TM) + broadband constant-angle AngleSweep",
    ),
    3: (
        "staircase_vs_conformal_convergence",
        "Staircase vs Kottke Conformal resolution convergence study + near-field phasor maps",
    ),
}


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Tutorial 6.5: Slanted and Blazed In-Coupler Gratings"
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Run fast, reduced-grid validation (< 1 min).",
    )
    parser.add_argument(
        "--full",
        action="store_true",
        help="Run full-resolution benchmark sweeps.",
    )
    parser.add_argument(
        "--task",
        type=int,
        choices=sorted(TASKS.keys()),
        default=None,
        help="Run a single task index (0..3) and save its data.",
    )
    parser.add_argument(
        "--list-tasks",
        action="store_true",
        help="List available tasks and exit.",
    )
    parser.add_argument(
        "--plot",
        action="store_true",
        help="Regenerate figures from cached data/ without re-running FDTD.",
    )
    parser.add_argument(
        "--out",
        type=Path,
        default=TUTORIAL_DIR / "data",
        help="Output directory for cached .npz and .json data.",
    )
    parser.add_argument(
        "--fig-dir",
        type=Path,
        default=TUTORIAL_DIR / "figures",
        help="Output directory for figures.",
    )
    parser.add_argument(
        "--processes",
        type=int,
        default=2,
        help="Number of worker processes for AngleSweep (default: 2).",
    )
    if argv is None:
        env_args = os.environ.get("WAKIS_TUTORIAL_ARGS", "").strip()
        argv = shlex.split(env_args) if env_args else sys.argv[1:]
    args, _ = parser.parse_known_args(argv)
    if not args.full:
        args.quick = True
    return args

## 1. Unit-Cell Setup Schematics & Staircase vs Conformal Mesh Views (Figures 00a/00b & Figure 1)

Before running the parameter sweeps, we inspect the 2D `FloquetCell` / `SolverFIT3D` simulation setups generated directly from the live solver instances for the slanted-ridge and sawtooth-blazed in-coupler profiles (note: the multi-layer Fourier Modal Method reference in `rcwa_multilayer_1d` and the Bragg slant formula are pure frequency-domain / analytical calculations and do not use a time-domain FIT grid):

### Primary Setup — Slanted $\text{TiO}_2$ Ridge In-Coupler ($\alpha = 25^\circ \approx \alpha_{\text{Bragg}},\, f = 0.55,\, d = 320\;\text{nm}$)

The primary setup models a single periodic unit cell ($\Lambda = 400\;\text{nm}$, depth $d = 320\;\text{nm}$, fill factor $f = 0.55$, slant angle $\alpha = 25^\circ$) separating air ($n = 1.00$) from the N-LAF2 glass waveguide substrate ($n_s = 1.74$, $\varepsilon_r = 3.03$) with Kottke conformal subpixel permittivity smoothing, periodic/Bloch boundaries on $\pm x$, and 14-cell CPML absorbers on $\pm z$.

![Slanted In-Coupler Geometry and Boundaries](figures/00a_setup_geometry.png)

![Slanted In-Coupler FIT Mesh and Excitation](figures/00b_setup_mesh_and_excitation.png)

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 400 nm × 9.14 nm × 1.55 µm (z ∈ [-613 nm, 933 nm]) |
| Domain | Grid cells | Nx × Ny × Nz = 43 × 1 × 169 (7267 cells) |
| Discretization | Δz & resolution | Δ_min = 9.14 nm, 24.4 cells/λ (in n_max = 2.35), 19.7 cells across feature (180 nm) |
| Boundary | -x face | Periodic |
| Boundary | +x face | Periodic |
| Boundary | -y face | Periodic |
| Boundary | +y face | Periodic |
| Boundary | -z face | CPML (14 cells, 128 nm) |
| Boundary | +z face | CPML (14 cells, 128 nm) |
| Material | Background | Air cladding / groove (n = 1.00, ε_r = 1.00) (isotropic) |
| Material | Region 1: N-LAF2 glass waveguide substrate (n_s = 1.74, ε_r = 3.03) | ε_r = 3.03, μ_r = 1.00, model = Isotropic dielectric, z ∈ [320 nm, 933 nm] (d = 613 nm) |
| Material | Region 2: Slanted TiO₂ ridge (α = 25° ≈ α_Bragg, f = 0.55, d = 320 nm, n = 2.35, ε_r = 5.52, Kottke subpixel) | ε_r = 5.52, μ_r = 1.00, model = Isotropic dielectric, z ∈ [0 m, 320 nm] (d = 320 nm) |
| Source | 1 · CurrentSource | pos: z = -374 nm (x ∈ [−200, +200] nm), pol: Ey (TE, θ_in = 0° normal & oblique AngleSweep), pulse: f₀ = 571.0 THz, Δf = 68.52 THz |
| Monitor | 2 · FloquetPort | Port 1 Reflected FloquetPort at z = -193 nm (R_m in air), pos: z = -193 nm (in air), freqs: f₀ = 571.0 THz (λ₀ = 525 nm) |
| Monitor | 3 · FloquetPort | Port 2 Transmitted FloquetPort at z = +537 nm (T₋₁ ≫ T₊₁ in glass), pos: z = +537 nm (in N-LAF2 glass), freqs: f₀ = 571.0 THz (λ₀ = 525 nm) |
| Variation | Slant angle & profile comparison (Task 0) | α ∈ [0°, 40°] (α_Bragg = 24.5°) and Binary (α = 0°) vs Sawtooth Blazed vs Slanted (α = 25°) |
| Variation | Groove depth & fill factor sweeps (Task 1) | d ∈ [100, 500] nm and f ∈ [0.25, 0.80] vs 28-slice multi-layer RCWA |
| Variation | Oblique FOV & broadband AngleSweep (Task 2) | θ_in ∈ [−20°, +20°] (Bloch ±x, TE & TM) and broadband λ ∈ [490, 565] nm at θ_in = 8° |
| Variation | Staircase vs Kottke conformal convergence (Task 3) | cpw ∈ {14, 18, 24, 32} cells/λ comparing staircase point-sampling vs 16×16 Kottke tensor smoothing |

### Companion Setup — Sawtooth Blazed $\text{TiO}_2$ In-Coupler under Oblique Floquet–Bloch Illumination ($\theta_{\text{in}} = 8^\circ$)

The companion setup models the right-angled triangular sawtooth blazed tooth ($x_{\text{fill}}(z) = f(1 - z/d)$, $f = 0.85$, $d = 320\;\text{nm}$) under oblique Floquet–Bloch illumination ($\theta_{\text{in}} = 8^\circ$, $k_x = -k_0 \sin\theta_{\text{in}}$ on the $\pm x$ Bloch faces).

![Sawtooth Blazed Geometry and Boundaries](figures/00a_setup_geometry_blazed.png)

![Sawtooth Blazed FIT Mesh and Excitation](figures/00b_setup_mesh_and_excitation_blazed.png)

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 400 nm × 9.14 nm × 1.55 µm (z ∈ [-613 nm, 933 nm]) |
| Domain | Grid cells | Nx × Ny × Nz = 43 × 1 × 169 (7267 cells) |
| Discretization | Δz & resolution | Δ_min = 9.14 nm, 24.4 cells/λ (in n_max = 2.35), 6.6 cells across feature (60 nm) |
| Boundary | -x face | Bloch (k_x = -1.67e+06 rad/m) |
| Boundary | +x face | Bloch (k_x = -1.67e+06 rad/m) |
| Boundary | -y face | Periodic |
| Boundary | +y face | Periodic |
| Boundary | -z face | CPML (14 cells, 128 nm) |
| Boundary | +z face | CPML (14 cells, 128 nm) |
| Material | Background | Air cladding / groove (n = 1.00, ε_r = 1.00) (isotropic) |
| Material | Region 1: N-LAF2 glass waveguide substrate (n_s = 1.74, ε_r = 3.03) | ε_r = 3.03, μ_r = 1.00, model = Isotropic dielectric, z ∈ [320 nm, 933 nm] (d = 613 nm) |
| Material | Region 2: Sawtooth blazed TiO₂ tooth (f = 0.85, d = 320 nm, n = 2.35, ε_r = 5.52, Kottke subpixel) | ε_r = 5.52, μ_r = 1.00, model = Isotropic dielectric, z ∈ [0 m, 320 nm] (d = 320 nm) |
| Source | 1 · CurrentSource | pos: z = -374 nm (x ∈ [−200, +200] nm), pol: Ey (TE, θ_in = 8°), pulse: f₀ = 571.0 THz, Δf = 68.52 THz |
| Monitor | 2 · FloquetPort | Port 1 Reflected FloquetPort at z = -193 nm (R_m in air), pos: z = -193 nm (in air), freqs: f₀ = 571.0 THz (λ₀ = 525 nm) |
| Monitor | 3 · FloquetPort | Port 2 Transmitted FloquetPort at z = +537 nm (T₋₁ > T₊₁ in glass), pos: z = +537 nm (in N-LAF2 glass), freqs: f₀ = 571.0 THz (λ₀ = 525 nm) |
| Variation | Sawtooth blazed vs binary & slanted (Tasks 0 & 2) | Right-angled triangular tooth (x_fill(z) = f·(1 − z/d), f = 0.85) across θ_in ∈ [−20°, +20°] |

Figure 1 below compares all three tooth profiles side-by-side along with the staircase vs Kottke conformal subpixel permittivity maps $\tilde{\varepsilon}_{xx}(x, z)$:
- **Panel (a):** The 2D `FloquetCell` domain geometry ($x \in [-\Lambda/2, +\Lambda/2]$ with Bloch boundaries, $z \in [z_{\min}, z_{\max}]$ with 14-cell CPML absorbers, tilted plane-wave source at $z_{\text{src}}$, reflection and transmission `DiffractionMonitor` ports, incident air medium $\varepsilon_{\text{below}}=1.0$, and N-LAF2 glass substrate $\varepsilon_{\text{above}}=1.74^2=3.03$) for **Binary** ($\alpha=0^\circ$), **Sawtooth Blazed**, and **Slanted** ($\alpha=25^\circ$) ridges.
- **Panel (b):** The **Staircase** Yee mesh (`averaging="staircase"`) for the $\alpha=25^\circ$ slanted ridge at 20 cells per wavelength, showing jagged cell-by-cell steps along the tilted sidewalls.
- **Panel (c):** The **Kottke Conformal** subpixel-smoothed Yee mesh (`averaging="kottke"`) on the **exact same grid**, showing smooth subcell volume fractions and sidewall normal vector $\hat{\mathbf{n}} = (\cos\alpha, 0, -\sin\alpha)$.

In [ ]:
def plot_slanted_coupler_setup_figures(fig_dir: Path) -> dict[str, Path]:
    """Generate the two-figure setup layout via ``wakis.schematic.save_setup_figures``
    for the primary slanted ridge in-coupler (``suffix=''``) and companion sawtooth
    blazed in-coupler (``suffix='_blazed'``).
    """
    from matplotlib.axes._base import _TransformedBoundsLocator
    from scipy.constants import c as c_light
    from wakis import figstyle, schematic
    from wakis.sources import GaussianSource
    from wakis.templates.photonics import Box

    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    figstyle.apply_style()

    f0 = c_light / WL_DESIGN_M
    wf = GaussianSource(frequency=f0, fwidth=0.12 * f0)
    pulse_dict = {
        "f0": f0,
        "fwidth": 0.12 * f0,
        "t_peak": float(4.0 / (np.pi * 0.12 * f0)),
        "t_end": float(wf.end_time),
        "src_time": wf,
    }

    class _ToothPoly:
        def __init__(self, vertices: np.ndarray, material: float, name: str):
            self.vertices = np.asarray(vertices, dtype=float)
            self.material = material
            self.name = name

    # ------------------------------------------------------------------
    # 1. Primary setup: Slanted TiO2 ridge in-coupler (alpha = 25 deg, f = 0.55)
    # ------------------------------------------------------------------
    cell_sl, minfo_sl = build_slanted_floquet_cell(
        period=PERIOD_INCOUPLER_M,
        height=DEFAULT_DEPTH_M,
        wavelength_min=WL_DESIGN_M,
        profile_type="slanted",
        slant_deg=WANG_SLANT_DEG,
        fill_factor=0.55,
        averaging="kottke",
        cpw=24,
    )
    grid_sl = wakis.GridFIT3D(
        -0.5 * cell_sl.Lx,
        0.5 * cell_sl.Lx,
        -0.5 * cell_sl.Ly,
        0.5 * cell_sl.Ly,
        cell_sl.zmin,
        cell_sl.zmax,
        cell_sl.nx,
        cell_sl.ny,
        cell_sl.nz,
        verbose=0,
    )
    solver_sl = wakis.SolverFIT3D(
        grid_sl,
        bc_low=["periodic", "periodic", "cpml"],
        bc_high=["periodic", "periodic", "cpml"],
        n_pml=cell_sl.n_pml,
        dtype=np.complex128,
        cfln=cell_sl.cfl,
        verbose=0,
    )
    cell_sl.material_callback(solver_sl)

    sub_sl = Box(
        center=(0.0, 0.0, 0.5 * (DEFAULT_DEPTH_M + cell_sl.zmax)),
        size=(cell_sl.Lx, cell_sl.Ly, cell_sl.zmax - DEFAULT_DEPTH_M),
        material=N_SUBSTRATE**2,
    )
    sub_sl.name = "N-LAF2 glass waveguide substrate (n_s = 1.74, ε_r = 3.03)"
    ridge_box_sl = Box(
        center=(0.0, 0.0, 0.5 * DEFAULT_DEPTH_M),
        size=(0.55 * cell_sl.Lx, cell_sl.Ly, DEFAULT_DEPTH_M),
        material=N_RIDGE**2,
    )
    ridge_lbl_sl = (
        "Slanted TiO₂ ridge (α = 25° ≈ α_Bragg, f = 0.55, d = 320 nm, n = 2.35, ε_r = 5.52, Kottke subpixel)"
    )
    ridge_box_sl.name = ridge_lbl_sl
    solver_sl._assigned_structures = (sub_sl, ridge_box_sl)

    mat_names_sl = {
        "background": "Air cladding / groove (n = 1.00, ε_r = 1.00)",
        "region_0": sub_sl.name,
        "region_1": ridge_lbl_sl,
    }
    sources_sl = [
        {
            "kind": "plane",
            "type_name": "CurrentSource",
            "short_word": "Plane Wave Jy",
            "label": f"Floquet CurrentSource Jy at z = {cell_sl.z_source*1e9:.0f} nm (λ₀ = 525 nm, +z)",
            "desc": f"Floquet CurrentSource Jy at z = {cell_sl.z_source*1e9:.0f} nm (TE / Ey, λ₀ = 525 nm, +z)",
            "pos_str": f"z = {cell_sl.z_source*1e9:.0f} nm (x ∈ [−200, +200] nm)",
            "polarization": "Ey (TE, θ_in = 0° normal & oblique AngleSweep)",
            "direction": 1,
            "h_bounds": (cell_sl.z_source, cell_sl.z_source),
            "v_bounds": (-0.5 * cell_sl.Lx, +0.5 * cell_sl.Lx),
            "pulse": pulse_dict,
        }
    ]
    monitors_sl = [
        {
            "kind": "plane",
            "type_name": "FloquetPort",
            "short_word": "Port 1 Refl.",
            "label": f"Port 1 Reflected FloquetPort at z = {cell_sl.z_refl*1e9:.0f} nm (R_m in air)",
            "desc": f"Port 1 Reflected FloquetPort / DiffractionMonitor (R_m, m ∈ [-2..+2]) at z = {cell_sl.z_refl*1e9:.0f} nm",
            "pos_str": f"z = {cell_sl.z_refl*1e9:.0f} nm (in air)",
            "freq_str": "f₀ = 571.0 THz (λ₀ = 525 nm)",
            "frequencies": np.array([f0]),
            "h_bounds": (cell_sl.z_refl, cell_sl.z_refl),
            "v_bounds": (-0.5 * cell_sl.Lx, +0.5 * cell_sl.Lx),
        },
        {
            "kind": "plane",
            "type_name": "FloquetPort",
            "short_word": "Port 2 Tran.",
            "label": f"Port 2 Transmitted FloquetPort at z = +{cell_sl.z_tran*1e9:.0f} nm (T₋₁ ≫ T₊₁ in glass)",
            "desc": f"Port 2 Transmitted FloquetPort / DiffractionMonitor (T_m in glass, T₋₁ > 70%, T₋₁/T₊₁ > 10×) at z = +{cell_sl.z_tran*1e9:.0f} nm",
            "pos_str": f"z = +{cell_sl.z_tran*1e9:.0f} nm (in N-LAF2 glass)",
            "freq_str": "f₀ = 571.0 THz (λ₀ = 525 nm)",
            "frequencies": np.array([f0]),
            "h_bounds": (cell_sl.z_tran, cell_sl.z_tran),
            "v_bounds": (-0.5 * cell_sl.Lx, +0.5 * cell_sl.Lx),
        },
    ]
    variations_sl = [
        {
            "name": "Slant angle & profile comparison (Task 0)",
            "details": "α ∈ [0°, 40°] (α_Bragg = 24.5°) and Binary (α = 0°) vs Sawtooth Blazed vs Slanted (α = 25°)",
        },
        {
            "name": "Groove depth & fill factor sweeps (Task 1)",
            "details": "d ∈ [100, 500] nm and f ∈ [0.25, 0.80] vs 28-slice multi-layer RCWA",
        },
        {
            "name": "Oblique FOV & broadband AngleSweep (Task 2)",
            "details": "θ_in ∈ [−20°, +20°] (Bloch ±x, TE & TM) and broadband λ ∈ [490, 565] nm at θ_in = 8°",
        },
        {
            "name": "Staircase vs Kottke conformal convergence (Task 3)",
            "details": "cpw ∈ {14, 18, 24, 32} cells/λ comparing staircase point-sampling vs 16×16 Kottke tensor smoothing",
        },
    ]

    res_sl = schematic.save_setup_figures(
        solver_sl,
        fig_dir,
        prefix="00",
        plane="zx",
        material_names=mat_names_sl,
        sources=sources_sl,
        monitors=monitors_sl,
        unit="nm",
        wavelength=WL_DESIGN_M,
        min_feature=minfo_sl["min_feature_nm"] * 1e-9,
        variations=variations_sl,
    )

    # Replace rectangular box on 00a with exact tilted parallelogram polygon in (z, x) coordinates
    w_sl = 0.55 * PERIOD_INCOUPLER_M
    shift_sl = DEFAULT_DEPTH_M * float(np.tan(np.deg2rad(WANG_SLANT_DEG)))
    x_top_0 = -0.5 * (w_sl - shift_sl)
    x_top_1 = x_top_0 + w_sl
    verts_sl = np.array(
        [
            [0.0, x_top_0],
            [DEFAULT_DEPTH_M, x_top_0 - shift_sl],
            [DEFAULT_DEPTH_M, x_top_1 - shift_sl],
            [0.0, x_top_1],
        ],
        dtype=float,
    )
    poly_sl = _ToothPoly(verts_sl, N_RIDGE**2, ridge_lbl_sl)
    solver_sl._assigned_structures = (sub_sl, poly_sl)

    fig_a_sl, ax_a_sl = schematic.plot_schematic(
        solver_sl,
        plane="zx",
        material_names=mat_names_sl,
        sources=sources_sl,
        monitors=monitors_sl,
        unit="nm",
        wavelength=WL_DESIGN_M,
    )
    for txt in ax_a_sl.texts:
        t_str = txt.get_text()
        if "1 · Plane" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_sl.z_source * 1e9, -145.0))
        elif "2 · Port 1" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_sl.z_refl * 1e9 + 18.0, +145.0))
        elif "3 · Port 2" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_sl.z_tran * 1e9, +145.0))
    figstyle.save(fig_a_sl, res_sl["geometry_figure"], close=True, auto_legend_below=False)

    solver_sl._assigned_structures = (sub_sl, ridge_box_sl)
    fig_b_sl, axes_b_sl = schematic.plot_mesh_and_excitation(
        solver_sl,
        plane="zx",
        material_names=mat_names_sl,
        sources=sources_sl,
        monitors=monitors_sl,
        unit="nm",
        mesh_every=2,
        zoom_bounds=(-0.05e-6, +0.37e-6, -0.19e-6, +0.19e-6),
        wavelength=WL_DESIGN_M,
        min_feature=minfo_sl["min_feature_nm"] * 1e-9,
    )
    ax_mesh_sl = axes_b_sl[0]
    for ch_ax in getattr(ax_mesh_sl, "child_axes", []):
        ch_ax.set_axes_locator(
            _TransformedBoundsLocator([0.66, 0.10, 0.32, 0.80], ax_mesh_sl.transAxes)
        )
    figstyle.save(fig_b_sl, res_sl["mesh_figure"], close=True, auto_legend_below=False)

    # ------------------------------------------------------------------
    # 2. Companion setup (_blazed): Sawtooth blazed TiO2 in-coupler (f = 0.85, d = 320 nm)
    # ------------------------------------------------------------------
    cell_bl, minfo_bl = build_slanted_floquet_cell(
        period=PERIOD_INCOUPLER_M,
        height=DEFAULT_DEPTH_M,
        wavelength_min=WL_DESIGN_M,
        profile_type="blazed",
        slant_deg=0.0,
        fill_factor=0.85,
        averaging="kottke",
        cpw=24,
    )
    grid_bl = wakis.GridFIT3D(
        -0.5 * cell_bl.Lx,
        0.5 * cell_bl.Lx,
        -0.5 * cell_bl.Ly,
        0.5 * cell_bl.Ly,
        cell_bl.zmin,
        cell_bl.zmax,
        cell_bl.nx,
        cell_bl.ny,
        cell_bl.nz,
        verbose=0,
    )
    kx_bl = (2.0 * np.pi / WL_DESIGN_M) * float(np.sin(np.deg2rad(8.0)))
    solver_bl = wakis.SolverFIT3D(
        grid_bl,
        bc_low=["bloch", "periodic", "cpml"],
        bc_high=["bloch", "periodic", "cpml"],
        n_pml=cell_bl.n_pml,
        k_bloch=(-kx_bl, 0.0, 0.0),
        dtype=np.complex128,
        cfln=cell_bl.cfl,
        verbose=0,
    )
    cell_bl.material_callback(solver_bl)

    sub_bl = Box(
        center=(0.0, 0.0, 0.5 * (DEFAULT_DEPTH_M + cell_bl.zmax)),
        size=(cell_bl.Lx, cell_bl.Ly, cell_bl.zmax - DEFAULT_DEPTH_M),
        material=N_SUBSTRATE**2,
    )
    sub_bl.name = "N-LAF2 glass waveguide substrate (n_s = 1.74, ε_r = 3.03)"
    blazed_lbl = (
        "Sawtooth blazed TiO₂ tooth (f = 0.85, d = 320 nm, n = 2.35, ε_r = 5.52, Kottke subpixel)"
    )
    blazed_box = Box(
        center=(0.0, 0.0, 0.5 * DEFAULT_DEPTH_M),
        size=(0.85 * cell_bl.Lx, cell_bl.Ly, DEFAULT_DEPTH_M),
        material=N_RIDGE**2,
    )
    blazed_box.name = blazed_lbl
    solver_bl._assigned_structures = (sub_bl, blazed_box)

    mat_names_bl = {
        "background": "Air cladding / groove (n = 1.00, ε_r = 1.00)",
        "region_0": sub_bl.name,
        "region_1": blazed_lbl,
    }
    sources_bl = [
        {
            "kind": "plane",
            "type_name": "CurrentSource",
            "short_word": "Bloch Jy (+z)",
            "label": f"Oblique Floquet CurrentSource Jy at z = {cell_bl.z_source*1e9:.0f} nm (θ_in = 8°, +z)",
            "desc": f"Oblique Floquet CurrentSource Jy at z = {cell_bl.z_source*1e9:.0f} nm (θ_in = 8°, Δφ_x = k_x Λ)",
            "pos_str": f"z = {cell_bl.z_source*1e9:.0f} nm (x ∈ [−200, +200] nm)",
            "polarization": "Ey (TE, θ_in = 8°)",
            "direction": 1,
            "h_bounds": (cell_bl.z_source, cell_bl.z_source),
            "v_bounds": (-0.5 * cell_bl.Lx, +0.5 * cell_bl.Lx),
            "pulse": pulse_dict,
        }
    ]
    monitors_bl = [
        {
            "kind": "plane",
            "type_name": "FloquetPort",
            "short_word": "Port 1 Refl.",
            "label": f"Port 1 Reflected FloquetPort at z = {cell_bl.z_refl*1e9:.0f} nm (R_m in air)",
            "desc": f"Port 1 Reflected FloquetPort / DiffractionMonitor (R_m) at z = {cell_bl.z_refl*1e9:.0f} nm",
            "pos_str": f"z = {cell_bl.z_refl*1e9:.0f} nm (in air)",
            "freq_str": "f₀ = 571.0 THz (λ₀ = 525 nm)",
            "frequencies": np.array([f0]),
            "h_bounds": (cell_bl.z_refl, cell_bl.z_refl),
            "v_bounds": (-0.5 * cell_bl.Lx, +0.5 * cell_bl.Lx),
        },
        {
            "kind": "plane",
            "type_name": "FloquetPort",
            "short_word": "Port 2 Tran.",
            "label": f"Port 2 Transmitted FloquetPort at z = +{cell_bl.z_tran*1e9:.0f} nm (T₋₁ > T₊₁ in glass)",
            "desc": f"Port 2 Transmitted FloquetPort / DiffractionMonitor (T_m in glass, T₋₁/T₊₁ > 8×) at z = +{cell_bl.z_tran*1e9:.0f} nm",
            "pos_str": f"z = +{cell_bl.z_tran*1e9:.0f} nm (in N-LAF2 glass)",
            "freq_str": "f₀ = 571.0 THz (λ₀ = 525 nm)",
            "frequencies": np.array([f0]),
            "h_bounds": (cell_bl.z_tran, cell_bl.z_tran),
            "v_bounds": (-0.5 * cell_bl.Lx, +0.5 * cell_bl.Lx),
        },
    ]

    res_bl = schematic.save_setup_figures(
        solver_bl,
        fig_dir,
        prefix="00",
        suffix="_blazed",
        plane="zx",
        material_names=mat_names_bl,
        sources=sources_bl,
        monitors=monitors_bl,
        unit="nm",
        wavelength=WL_DESIGN_M,
        min_feature=minfo_bl["min_feature_nm"] * 1e-9,
        variations=[
            {
                "name": "Sawtooth blazed vs binary & slanted (Tasks 0 & 2)",
                "details": "Right-angled triangular tooth (x_fill(z) = f·(1 − z/d), f = 0.85) across θ_in ∈ [−20°, +20°]",
            },
        ],
    )

    w_bl = 0.85 * PERIOD_INCOUPLER_M
    verts_bl = np.array(
        [
            [0.0, -0.5 * w_bl],
            [DEFAULT_DEPTH_M, -0.5 * w_bl],
            [0.0, +0.5 * w_bl],
        ],
        dtype=float,
    )
    poly_bl = _ToothPoly(verts_bl, N_RIDGE**2, blazed_lbl)
    solver_bl._assigned_structures = (sub_bl, poly_bl)

    fig_a_bl, ax_a_bl = schematic.plot_schematic(
        solver_bl,
        plane="zx",
        material_names=mat_names_bl,
        sources=sources_bl,
        monitors=monitors_bl,
        unit="nm",
        wavelength=WL_DESIGN_M,
    )
    for txt in ax_a_bl.texts:
        t_str = txt.get_text()
        if "1 · Bloch" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_bl.z_source * 1e9, -145.0))
        elif "2 · Port 1" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_bl.z_refl * 1e9 + 18.0, +145.0))
        elif "3 · Port 2" in t_str:
            txt.set_ha("center")
            txt.set_position((cell_bl.z_tran * 1e9, +145.0))
    figstyle.save(fig_a_bl, res_bl["geometry_figure"], close=True, auto_legend_below=False)

    solver_bl._assigned_structures = (sub_bl, blazed_box)
    fig_b_bl, axes_b_bl = schematic.plot_mesh_and_excitation(
        solver_bl,
        plane="zx",
        material_names=mat_names_bl,
        sources=sources_bl,
        monitors=monitors_bl,
        unit="nm",
        mesh_every=2,
        zoom_bounds=(-0.05e-6, +0.37e-6, -0.19e-6, +0.19e-6),
        wavelength=WL_DESIGN_M,
        min_feature=minfo_bl["min_feature_nm"] * 1e-9,
    )
    ax_mesh_bl = axes_b_bl[0]
    for ch_ax in getattr(ax_mesh_bl, "child_axes", []):
        ch_ax.set_axes_locator(
            _TransformedBoundsLocator([0.66, 0.10, 0.32, 0.80], ax_mesh_bl.transAxes)
        )
    figstyle.save(fig_b_bl, res_bl["mesh_figure"], close=True, auto_legend_below=False)

    return {"slanted": res_sl, "blazed": res_bl}


def plot_fig1_schematic_and_meshes(fig_dir: Path) -> Path:
    """Generate Figure 1: Setup schematic + Staircase vs Kottke Conformal mesh comparison."""
    plot_slanted_coupler_setup_figures(fig_dir)
    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.6), constrained_layout=True)

    # --- Panel (a): Binary vs Blazed vs Slanted Unit-Cell Schematic ---
    ax = axes[0]
    ax.set_xlim(-0.55, 3.05)
    ax.set_ylim(-1.15, 1.35)
    ax.set_xlabel(r"Unit-cell profiles along $x / \Lambda$ ($\Lambda = 400$ nm)")
    ax.set_ylabel(r"Vertical coordinate $z$ [$\mu$m]")
    ax.set_title("(a) Binary, Blazed & Slanted Floquet Unit Cells")

    # Background air & glass + CPML bands
    ax.add_patch(Rectangle((-0.45, -0.90), 3.40, 0.90, facecolor="#F7FAFC", edgecolor="none", zorder=1))
    ax.add_patch(Rectangle((-0.45, 0.0), 3.40, 1.05, facecolor="#DCEEFB", edgecolor="none", zorder=1))
    ax.add_patch(
        Rectangle((-0.45, -1.12), 3.40, 0.22, facecolor="#DDDDDD", edgecolor="#666666", hatch="///", lw=0.8, zorder=2)
    )
    ax.add_patch(
        Rectangle((-0.45, 1.05), 3.40, 0.22, facecolor="#DDDDDD", edgecolor="#666666", hatch="///", lw=0.8, zorder=2)
    )
    ax.text(1.25, -1.01, "Bottom CPML (14 cells, -z)", ha="center", va="center", fontsize=7.9, fontweight="bold")
    ax.text(1.25, 1.16, "Top CPML (14 cells, +z)", ha="center", va="center", fontsize=7.9, fontweight="bold")

    # 1. Binary tooth centered at x = 0.0
    ax.add_patch(Rectangle((-0.25, 0.0), 0.50, 0.34, facecolor=PALETTE["sky"], edgecolor="#004C80", lw=1.3, zorder=3))
    ax.text(0.0, 0.40, "Binary\n($\\alpha=0^\\circ$)", ha="center", fontsize=8.0, fontweight="bold", color="#004C80")
    ax.annotate("", xy=(-0.26, 0.85), xytext=(0.0, 0.34), arrowprops=dict(arrowstyle="->", color=PALETTE["B"], lw=1.5))
    ax.annotate("", xy=(+0.26, 0.85), xytext=(0.0, 0.34), arrowprops=dict(arrowstyle="->", color=PALETTE["B"], lw=1.5))
    ax.text(0.0, 0.88, "$T_{-1}=T_{+1}$", ha="center", fontsize=7.8, color=PALETTE["B"])

    # 2. Sawtooth Blazed tooth centered around x = 1.25
    blazed_poly = Polygon(
        [(0.90, 0.0), (1.65, 0.0), (0.90, 0.34)],
        closed=True,
        facecolor=PALETTE["gold"],
        edgecolor="#805000",
        lw=1.3,
        zorder=3,
    )
    ax.add_patch(blazed_poly)
    ax.text(1.25, 0.40, "Blazed\n(Sawtooth)", ha="center", fontsize=8.0, fontweight="bold", color="#805000")
    ax.annotate("", xy=(0.95, 0.86), xytext=(1.25, 0.25), arrowprops=dict(arrowstyle="->", color="#805000", lw=1.8))
    ax.text(1.25, 0.88, "$T_{-1} > T_{+1}$", ha="center", fontsize=7.8, color="#805000")

    # 3. Slanted parallelogram tooth centered around x = 2.50
    shift_x = 0.34 * np.tan(np.deg2rad(25.0))
    slanted_poly = Polygon(
        [(2.18, 0.0), (2.73, 0.0), (2.73 + shift_x, 0.34), (2.18 + shift_x, 0.34)],
        closed=True,
        facecolor=PALETTE["R"],
        edgecolor="#802000",
        lw=1.3,
        zorder=3,
    )
    ax.add_patch(slanted_poly)
    ax.text(2.52, 0.40, "Slanted\n($\\alpha=25^\\circ$)", ha="center", fontsize=8.0, fontweight="bold", color=PALETTE["R"])
    ax.annotate("", xy=(2.15, 0.88), xytext=(2.52, 0.34), arrowprops=dict(arrowstyle="->", color=PALETTE["R"], lw=2.2))
    ax.text(2.45, 0.88, "$T_{-1} \\gg T_{+1}$", ha="center", fontsize=8.0, fontweight="bold", color=PALETTE["R"])

    # Bloch boundaries, Source & Monitor planes
    for xb in (-0.45, 2.95):
        ax.plot([xb, xb], [-1.12, 1.27], color=PALETTE["purple"], lw=1.8, ls="-.")
    ax.axhline(-0.62, color=PALETTE["R"], lw=1.6, ls="-")
    ax.text(1.25, -0.74, r"Source $J_y(x,t)=s(t)e^{-ik_x x}$ ($z_{\mathrm{src}}$)", ha="center", fontsize=7.9, color=PALETTE["R"])
    ax.axhline(-0.32, color=PALETTE["B"], lw=1.4, ls="--")
    ax.text(1.25, -0.23, r"Port 1: Refl. Monitor ($z_{\mathrm{refl}}$, Air $n=1.0$)", ha="center", fontsize=7.8, color=PALETTE["B"])
    ax.axhline(0.64, color=PALETTE["G"], lw=1.4, ls="--")
    ax.text(1.25, 0.68, r"Port 2: Tran. Monitor ($z_{\mathrm{tran}}$, Glass $n_s=1.74$)", ha="center", fontsize=7.8, color=PALETTE["G"])

    # --- Panels (b) & (c): Staircase vs Kottke Conformal Mesh Views at Equal Cells ---
    for idx_p, (avg_mode, title_str) in enumerate(
        [
            ("staircase", r"(b) Staircase Mesh ($\mathrm{averaging='staircase'}$)"),
            ("kottke", r"(c) Conformal Subpixel Mesh ($\mathrm{averaging='kottke'}$)"),
        ],
        start=1,
    ):
        ax = axes[idx_p]
        cell_m, minfo = build_slanted_floquet_cell(
            period=PERIOD_INCOUPLER_M,
            height=DEFAULT_DEPTH_M,
            wavelength_min=WL_DESIGN_M,
            profile_type="slanted",
            slant_deg=25.0,
            fill_factor=0.55,
            averaging=avg_mode,
            cpw=18,
        )
        grid = wakis.GridFIT3D(
            -0.5 * cell_m.Lx,
            0.5 * cell_m.Lx,
            -0.5 * cell_m.Ly,
            0.5 * cell_m.Ly,
            cell_m.zmin,
            cell_m.zmax,
            cell_m.nx,
            cell_m.ny,
            cell_m.nz,
            verbose=0,
        )
        solver = wakis.SolverFIT3D(
            grid,
            bc_low=["bloch", "bloch", "cpml"],
            bc_high=["bloch", "bloch", "cpml"],
            n_pml=cell_m.n_pml,
            dtype=np.complex128,
            verbose=0,
        )
        cell_m.material_callback(solver)
        eps_x = 1.0 / (np.asarray(solver.ieps[:, 0, :, "x"], dtype=float) * epsilon_0)

        x_nm = grid.x * 1e9
        z_nm = grid.z * 1e9
        z_mask = (z_nm[:-1] >= -90.0) & (z_nm[:-1] <= 410.0)
        iz = np.where(z_mask)[0]
        z_sub = z_nm[iz[0] : iz[-1] + 2]

        pcm = ax.pcolormesh(
            x_nm,
            z_sub,
            eps_x[:, iz].T,
            cmap="YlOrBr",
            vmin=1.0,
            vmax=N_RIDGE**2,
            shading="flat",
        )
        cbar = fig.colorbar(pcm, ax=ax, pad=0.02, shrink=0.92)
        cbar.set_label(r"Effective cell permittivity $\tilde{\varepsilon}_{xx}(x,z)$")

        # Draw Yee grid lines
        vlines = [((xv, z_sub[0]), (xv, z_sub[-1])) for xv in x_nm]
        hlines = [((x_nm[0], zv), (x_nm[-1], zv)) for zv in z_sub]
        ax.add_collection(LineCollection(vlines, colors="black", linewidths=0.35, alpha=0.45))
        ax.add_collection(LineCollection(hlines, colors="black", linewidths=0.35, alpha=0.45))

        # Overlay exact analytical slanted tooth boundary
        z_line = np.linspace(0.0, DEFAULT_DEPTH_M * 1e9, 200)
        tan_a = np.tan(np.deg2rad(25.0))
        # Ridge spans u in [0, 0.55) in periodic coordinates
        for x0_edge in (-200.0, -200.0 + 0.55 * 400.0, 0.0, 0.55 * 400.0 - 400.0):
            x_wall = ((x0_edge + z_line * tan_a + 200.0) % 400.0) - 200.0
            # Plot continuous segments without wrap jumps
            jumps = np.where(np.abs(np.diff(x_wall)) > 150.0)[0] + 1
            for seg_x, seg_z in zip(np.split(x_wall, jumps), np.split(z_line, jumps)):
                ax.plot(seg_x, seg_z, color=PALETTE["B"], lw=1.8, ls="--")

        ax.axhline(0.0, color="black", lw=1.1, ls=":")
        ax.axhline(DEFAULT_DEPTH_M * 1e9, color="black", lw=1.1, ls=":")

        if avg_mode == "kottke":
            # Draw sidewall unit normal vector n_hat = (cos alpha, -sin alpha)
            ax.annotate(
                "",
                xy=(-40.0 + 55.0 * np.cos(np.deg2rad(25.0)), 160.0 - 55.0 * np.sin(np.deg2rad(25.0))),
                xytext=(-40.0, 160.0),
                arrowprops=dict(arrowstyle="->", color=PALETTE["B"], lw=2.2),
            )
            ax.text(
                10.0,
                125.0,
                r"$\hat{\mathbf{n}}=(\cos\alpha,-\sin\alpha)$",
                color=PALETTE["B"],
                fontsize=8.0,
                fontweight="bold",
                bbox=dict(fc="white", ec=PALETTE["B"], alpha=0.9, pad=1.5),
            )

        ax.text(
            0.0,
            -55.0,
            f"Air ($\\varepsilon_r=1.0$) | Grid: {minfo['nx']}x{minfo['nz']} ({minfo['dx_nm']:.1f} nm)",
            ha="center",
            fontsize=7.9,
            bbox=dict(fc="white", alpha=0.88, pad=1.5),
        )
        ax.text(
            0.0,
            365.0,
            r"N-LAF2 Glass Substrate ($\varepsilon_r = 3.03$)",
            ha="center",
            fontsize=7.9,
            bbox=dict(fc="white", alpha=0.88, pad=1.5),
        )
        ax.set_xlabel(r"$x$ [nm]")
        ax.set_ylabel(r"$z$ [nm]")
        ax.set_title(title_str)

    fig_dir.mkdir(parents=True, exist_ok=True)
    out_path = fig_dir / "01_slanted_binary_blazed_schematic_and_mesh.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path

## 2. Figure Generation Functions (Figures 2–6)

Figures 2–6 compare Wakis 2D FIT against multi-layer RCWA across slant angle $\alpha$,
groove depth $d$, fill factor $f$, incidence angle $\theta_{\text{in}}$, wavelength $\lambda_0$,
and grid resolution.

In [ ]:
def plot_fig2_slant_and_profiles(data_t0: dict[str, np.ndarray], fig_dir: Path) -> Path:
    """Generate Figure 2: Slant angle sweep, +/-1 selectivity ratio, and Binary vs Blazed vs Slanted."""
    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.5), constrained_layout=True)

    # --- Panel (a): Diffraction Efficiency vs Slant Angle alpha ---
    ax = axes[0]
    s_rcwa = data_t0["slant_rcwa_deg"]
    s_wakis = data_t0["slant_wakis_deg"]

    ax.plot(
        s_rcwa,
        data_t0["rcwa_t_m1"] * 100.0,
        color=PALETTE["R"],
        lw=2.0,
        label=r"$T_{-1}$ (Guided order) RCWA",
    )
    ax.plot(
        s_wakis,
        data_t0["wakis_t_m1"] * 100.0,
        color=PALETTE["R"],
        marker="o",
        ls="none",
        ms=6.5,
        label=r"$T_{-1}$ Wakis FIT",
    )

    ax.plot(
        s_rcwa,
        data_t0["rcwa_t_p1"] * 100.0,
        color=PALETTE["B"],
        lw=1.9,
        ls="--",
        label=r"$T_{+1}$ (Parasitic order) RCWA",
    )
    ax.plot(
        s_wakis,
        data_t0["wakis_t_p1"] * 100.0,
        color=PALETTE["B"],
        marker="s",
        ls="none",
        ms=6.0,
        label=r"$T_{+1}$ Wakis FIT",
    )

    ax.plot(
        s_rcwa,
        data_t0["rcwa_t_0"] * 100.0,
        color=PALETTE["G"],
        lw=1.7,
        ls="-.",
        label=r"$T_0$ (0th-order see-through) RCWA",
    )
    ax.plot(
        s_wakis,
        data_t0["wakis_t_0"] * 100.0,
        color=PALETTE["G"],
        marker="^",
        ls="none",
        ms=5.8,
        label=r"$T_0$ Wakis FIT",
    )

    alpha_bragg = bragg_slant_angle_deg(0.0)
    ax.axvline(alpha_bragg, color=PALETTE["gold"], lw=1.5, ls=":")
    ax.text(
        alpha_bragg + 1.0,
        52.0,
        f"Bragg slant\n$\\alpha_{{\\mathrm{{Bragg}}}}={alpha_bragg:.1f}^\\circ$\n(Wang & Ma $25^\\circ$)",
        fontsize=7.8,
        color="#8C5800",
        fontweight="bold",
    )

    ax.set_xlabel(r"Groove slant angle $\alpha$ [deg]")
    ax.set_ylabel("Transmitted diffraction efficiency [%]")
    ax.set_xlim(0.0, 42.0)
    ax.set_ylim(0.0, 78.0)
    ax.set_title(r"(a) Order Efficiencies vs Slant Angle $\alpha$ ($\theta_{\mathrm{in}}=0^\circ$)")
    ax.legend(loc="upper left", fontsize=7.4, framealpha=0.92)

    # --- Panel (b): Directionality Selectivity Ratio T_{-1}/T_{+1} & Energy Conservation ---
    ax = axes[1]
    ratio_rcwa = data_t0["rcwa_t_m1"] / np.maximum(data_t0["rcwa_t_p1"], 1e-6)
    ratio_wakis = data_t0["wakis_t_m1"] / np.maximum(data_t0["wakis_t_p1"], 1e-6)

    ax.plot(s_rcwa, ratio_rcwa, color=PALETTE["purple"], lw=2.1, label=r"Selectivity $T_{-1}/T_{+1}$ RCWA")
    ax.plot(s_wakis, ratio_wakis, color=PALETTE["purple"], marker="D", ls="none", ms=6.2, label=r"Selectivity $T_{-1}/T_{+1}$ Wakis")
    ax.axhline(1.0, color=PALETTE["grey"], ls="--", lw=1.2, label=r"Symmetric limit ($T_{-1}=T_{+1}$ at $\alpha=0^\circ$)")

    ax2 = ax.twinx()
    ax2.plot(
        s_wakis,
        data_t0["wakis_energy"] * 100.0,
        color=PALETTE["ink"],
        marker="o",
        ls=":",
        lw=1.5,
        label=r"Total energy $\sum(R_m+T_m)$ [%]",
    )
    ax2.set_ylabel(r"Total energy $\sum_m (R_m + T_m)$ [%]", color=PALETTE["ink"])
    ax2.set_ylim(95.0, 105.0)
    ax2.grid(False)

    ax.set_xlabel(r"Groove slant angle $\alpha$ [deg]")
    ax.set_ylabel(r"Guided-to-parasitic ratio $T_{-1} / T_{+1}$", color=PALETTE["purple"])
    ax.set_xlim(0.0, 42.0)
    ax.set_ylim(0.0, max(float(np.max(ratio_rcwa)), float(np.max(ratio_wakis))) * 1.15)
    ax.set_title(r"(b) Unidirectional Blazing Ratio & Passivity")
    ax.legend(loc="upper left", fontsize=7.6, framealpha=0.92)

    # --- Panel (c): Binary vs Sawtooth Blazed vs Slanted Order Spectrum ---
    ax = axes[2]
    orders = data_t0["orders"]
    x_pos = np.arange(orders.size)
    w_bar = 0.26

    specs = [
        ("binary", r"Binary ($\alpha=0^\circ$)", PALETTE["sky"], -w_bar),
        ("blazed", "Blazed (Sawtooth)", PALETTE["gold"], 0.0),
        ("slanted", r"Slanted ($\alpha=25^\circ$)", PALETTE["R"], +w_bar),
    ]
    for p_key, lbl, col_b, dx_b in specs:
        w_vals = np.nan_to_num(data_t0[f"prof_{p_key}_wakis_t"], nan=0.0) * 100.0
        r_vals = np.nan_to_num(data_t0[f"prof_{p_key}_rcwa_t"], nan=0.0) * 100.0
        ax.bar(x_pos + dx_b, w_vals, width=w_bar, color=col_b, edgecolor="black", lw=0.7, label=f"{lbl} Wakis")
        ax.plot(x_pos + dx_b, r_vals, marker="d", color="black", ls="none", ms=5.0)

    ax.plot([], [], marker="d", color="black", ls="none", label="RCWA reference")
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"$m={m:+d}$" if m != 0 else "$m=0$" for m in orders])
    ax.set_xlabel("Transmitted diffraction order $m$ in waveguide")
    ax.set_ylabel("Transmitted efficiency $T_m$ [%]")
    ax.set_ylim(0.0, 76.0)
    ax.set_title("(c) Binary vs Blazed vs Slanted Order Distribution")
    ax.legend(loc="upper right", fontsize=7.4, framealpha=0.92)

    out_path = fig_dir / "02_efficiency_vs_slant_and_profile_comparison.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path


def plot_fig3_depth_and_fill(data_t1: dict[str, np.ndarray], fig_dir: Path) -> Path:
    """Generate Figure 3: Groove depth sweep, fill factor sweep, and 2D RCWA design map."""
    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.5), constrained_layout=True)

    # --- Panel (a): Efficiency vs Groove Depth d (at alpha=25 deg, fill=0.66) ---
    ax = axes[0]
    d_r = data_t1["depths_r_nm"]
    d_w = data_t1["depths_w_nm"]

    ax.plot(d_r, data_t1["rcwa_depth_m1"] * 100.0, color=PALETTE["R"], lw=2.0, label=r"$T_{-1}$ (Guided) RCWA")
    ax.plot(d_w, data_t1["wakis_depth_m1"] * 100.0, color=PALETTE["R"], marker="o", ls="none", ms=6.2, label=r"$T_{-1}$ Wakis FIT")

    ax.plot(d_r, data_t1["rcwa_depth_0"] * 100.0, color=PALETTE["G"], lw=1.8, ls="--", label=r"$T_0$ (0th order) RCWA")
    ax.plot(d_w, data_t1["wakis_depth_0"] * 100.0, color=PALETTE["G"], marker="s", ls="none", ms=5.8, label=r"$T_0$ Wakis FIT")

    ax.plot(d_r, data_t1["rcwa_depth_p1"] * 100.0, color=PALETTE["B"], lw=1.8, ls="-.", label=r"$T_{+1}$ (Parasitic) RCWA")
    ax.plot(d_w, data_t1["wakis_depth_p1"] * 100.0, color=PALETTE["B"], marker="^", ls="none", ms=5.8, label=r"$T_{+1}$ Wakis FIT")

    ax.axvline(WANG_DEPTH_M * 1e9, color=PALETTE["gold"], lw=1.5, ls=":")
    ax.text(
        WANG_DEPTH_M * 1e9 - 115.0,
        14.0,
        "Wang & Ma (2023)\nTable 1: $d=415$ nm",
        fontsize=7.8,
        color="#8C5800",
        fontweight="bold",
    )
    ax.set_xlabel(r"Groove depth $d$ [nm] ($\alpha=25^\circ$, $f=0.66$)")
    ax.set_ylabel("Transmitted diffraction efficiency [%]")
    ax.set_ylim(0.0, 78.0)
    ax.set_title(r"(a) Efficiency vs Groove Depth $d$: Wakis vs RCWA")
    ax.legend(loc="center left", fontsize=7.4, framealpha=0.92)

    # --- Panel (b): Efficiency vs Fill Factor f (at alpha=25 deg, d=320 nm) ---
    ax = axes[1]
    f_r = data_t1["fills_r"]
    f_w = data_t1["fills_w"]

    ax.plot(f_r, data_t1["rcwa_fill_m1"] * 100.0, color=PALETTE["R"], lw=2.0, label=r"$T_{-1}$ (Guided) RCWA")
    ax.plot(f_w, data_t1["wakis_fill_m1"] * 100.0, color=PALETTE["R"], marker="o", ls="none", ms=6.2, label=r"$T_{-1}$ Wakis FIT")

    ax.plot(f_r, data_t1["rcwa_fill_p1"] * 100.0, color=PALETTE["B"], lw=1.8, ls="--", label=r"$T_{+1}$ (Parasitic) RCWA")
    ax.plot(f_w, data_t1["wakis_fill_p1"] * 100.0, color=PALETTE["B"], marker="s", ls="none", ms=6.0, label=r"$T_{+1}$ Wakis FIT")

    ax.axvline(WANG_FILL_FACTOR, color=PALETTE["gold"], lw=1.5, ls=":")
    ax.text(
        WANG_FILL_FACTOR - 0.24,
        22.0,
        "Wang & Ma (2023)\nTable 1: $f=0.66$",
        fontsize=7.8,
        color="#8C5800",
        fontweight="bold",
    )
    ax.set_xlabel(r"Ridge fill factor $f$ ($\alpha=25^\circ$, $d=320$ nm)")
    ax.set_ylabel("Transmitted diffraction efficiency [%]")
    ax.set_ylim(0.0, 78.0)
    ax.set_title(r"(b) Efficiency vs Fill Factor $f$: Wakis vs RCWA")
    ax.legend(loc="upper left", fontsize=7.6, framealpha=0.92)

    # --- Panel (c): 2D Design Map T_{-1}(f, d) at alpha=25 deg ---
    ax = axes[2]
    mf = data_t1["map_fills"]
    md = data_t1["map_depths_nm"]
    mt = data_t1["map_t_m1"] * 100.0

    pcm = ax.pcolormesh(mf, md, mt, cmap="viridis", shading="auto", vmin=15.0, vmax=72.0)
    cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
    cbar.set_label(r"Guided $m=-1$ Efficiency $T_{-1}$ [%]")

    # Mark Wang & Ma Table 1 (f=0.66, d=415 nm) and Levola (2007) (d=330 nm)
    ax.plot(WANG_FILL_FACTOR, WANG_DEPTH_M * 1e9, marker="*", color="white", mec="black", ms=12, zorder=5)
    ax.annotate(
        "Wang & Ma (2023)\nTable 1 ($f=0.66, d=415$ nm)",
        xy=(WANG_FILL_FACTOR, WANG_DEPTH_M * 1e9),
        xytext=(0.30, 395.0),
        color="white",
        fontsize=7.8,
        fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="white", lw=1.2),
    )
    ax.plot(0.55, 330.0, marker="D", color=PALETTE["gold"], mec="black", ms=8.5, zorder=5)
    ax.annotate(
        "Levola (2007)\n($d=330$ nm)",
        xy=(0.55, 330.0),
        xytext=(0.28, 250.0),
        color="white",
        fontsize=7.8,
        fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="white", lw=1.2),
    )

    ax.set_xlabel(r"Fill factor $f$")
    ax.set_ylabel(r"Groove depth $d$ [nm]")
    ax.set_title(r"(c) 2D Slanted Coupler Map $T_{-1}(f, d)$ ($\alpha=25^\circ$)")

    out_path = fig_dir / "03_depth_and_fill_factor_maps.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path


def plot_fig4_oblique_and_broadband(data_t2: dict[str, np.ndarray], fig_dir: Path) -> Path:
    """Generate Figure 4: Oblique FOV sweep, TE vs TM polarization, and broadband AngleSweep."""
    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.5), constrained_layout=True)

    # --- Panel (a): Oblique Incidence Sweep (Binary vs Blazed vs Slanted) ---
    ax = axes[0]
    ang = data_t2["angles_pos_deg"]
    specs = [
        ("slanted", r"Slanted ($\alpha=25^\circ$)", PALETTE["R"], "o", "-"),
        ("blazed", "Blazed (Sawtooth)", PALETTE["gold"], "s", "--"),
        ("binary", r"Binary ($\alpha=0^\circ$)", PALETTE["B"], "^", "-."),
    ]
    for p_key, lbl, col_p, mk, ls_p in specs:
        ax.plot(ang, data_t2[f"oblique_{p_key}_rcwa_m1"] * 100.0, color=col_p, ls=ls_p, lw=1.9, label=f"{lbl} RCWA")
        ax.plot(
            ang,
            data_t2[f"oblique_{p_key}_wakis_m1"] * 100.0,
            color=col_p,
            marker=mk,
            ls="none",
            ms=6.2,
            label=f"{lbl} Wakis",
        )

    ax.set_xlabel(r"Incident FOV angle $\theta_{\mathrm{in}}$ [deg]")
    ax.set_ylabel(r"Guided $m=-1$ efficiency $T_{-1}$ [%]")
    ax.set_ylim(0.0, 78.0)
    ax.set_title("(a) Oblique FOV Sweep: Slanted vs Blazed vs Binary")
    ax.legend(loc="center right", fontsize=7.3, framealpha=0.92)

    # --- Panel (b): TE vs TM Polarization for Slanted Grating ---
    ax = axes[1]
    ax.plot(
        ang,
        data_t2["oblique_slanted_rcwa_m1"] * 100.0,
        color=PALETTE["R"],
        lw=2.0,
        label=r"TE ($s$-pol) RCWA",
    )
    ax.plot(
        ang,
        data_t2["oblique_slanted_wakis_m1"] * 100.0,
        color=PALETTE["R"],
        marker="o",
        ls="none",
        ms=6.2,
        label=r"TE ($s$-pol) Wakis FIT",
    )

    ax.plot(
        ang,
        data_t2["oblique_slanted_tm_rcwa_m1"] * 100.0,
        color=PALETTE["purple"],
        lw=2.0,
        ls="--",
        label=r"TM ($p$-pol) RCWA",
    )
    ax.plot(
        ang,
        data_t2["oblique_slanted_tm_wakis_m1"] * 100.0,
        color=PALETTE["purple"],
        marker="D",
        ls="none",
        ms=6.2,
        label=r"TM ($p$-pol) Wakis FIT",
    )
    ax.plot(
        ang,
        data_t2["oblique_slanted_energy"] * 100.0,
        color=PALETTE["ink"],
        ls=":",
        marker="d",
        ms=4.5,
        label=r"Passivity $\sum(R_m+T_m)$",
    )

    ax.set_xlabel(r"Incident FOV angle $\theta_{\mathrm{in}}$ [deg]")
    ax.set_ylabel(r"Efficiency / Total energy [%]")
    ax.set_ylim(0.0, 106.0)
    ax.set_title(r"(b) Slanted Coupler ($\alpha=25^\circ$): TE vs TM Polarization")
    ax.legend(loc="center right", fontsize=7.5, framealpha=0.92)

    # --- Panel (c): Broadband Constant-Angle AngleSweep at theta_in = 8 deg ---
    ax = axes[2]
    wl_bb = data_t2["wl_bb_nm"]
    ax.plot(
        wl_bb,
        data_t2["rc_bb_8deg"] * 100.0,
        color=PALETTE["G"],
        lw=2.1,
        label=r"Exact constant-angle RCWA ($\theta_{\mathrm{in}}=8^\circ$)",
    )
    ax.plot(
        wl_bb,
        data_t2["w_bb_8deg"] * 100.0,
        color=PALETTE["G"],
        marker="o",
        ls="none",
        ms=6.5,
        label=r"Wakis $\mathrm{AngleSweep.resample()}$ ($\theta_{\mathrm{in}}=8^\circ$)",
    )
    ax.plot(
        wl_bb,
        data_t2["w_bb_8deg_shifted"] * 100.0,
        color=PALETTE["grey"],
        ls="--",
        marker="x",
        ms=6.0,
        label=r"Shifted $\mathrm{resample(index\_shift=1)}$ ($\theta=12^\circ$)",
    )

    ax.set_xlabel(r"Wavelength $\lambda_0$ [nm]")
    ax.set_ylabel(r"Guided $m=-1$ efficiency $T_{-1}(\lambda_0, \theta_{\mathrm{in}}=8^\circ)$ [%]")
    ax.set_ylim(15.0, 75.0)
    ax.set_title(r"(c) Broadband Constant-Angle Sweep ($\theta_{\mathrm{in}}=8^\circ$)")
    ax.legend(loc="lower left", fontsize=7.4, framealpha=0.92)

    out_path = fig_dir / "04_oblique_incidence_and_broadband_sweeps.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path


def plot_fig5_convergence_and_nearfields(data_t3: dict[str, np.ndarray], fig_dir: Path) -> Path:
    """Generate Figure 5: Staircase vs Conformal convergence and Binary vs Slanted near fields."""
    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.6), constrained_layout=True)

    # --- Panel (a): Staircase vs Kottke Conformal Convergence Study ---
    ax = axes[0]
    cpw_lv = data_t3["cpw_levels"]
    err_k_rms = data_t3["err_kottke_rms"]
    err_s_rms = data_t3["err_stair_rms"]
    err_k_m1 = data_t3["err_kottke_m1"]
    err_s_m1 = data_t3["err_stair_m1"]

    ax.loglog(
        cpw_lv,
        err_s_rms,
        marker="s",
        color=PALETTE["R"],
        lw=2.0,
        ls="--",
        label="Staircase RMS error (orders $-2..+2$)",
    )
    ax.loglog(
        cpw_lv,
        err_k_rms,
        marker="o",
        color=PALETTE["G"],
        lw=2.2,
        ls="-",
        label="Kottke Conformal RMS error",
    )
    ax.loglog(
        cpw_lv,
        err_s_m1,
        marker="^",
        color=PALETTE["gold"],
        lw=1.6,
        ls=":",
        label=r"Staircase $|T_{-1} - T_{-1}^{\mathrm{RCWA}}|$",
    )
    ax.loglog(
        cpw_lv,
        err_k_m1,
        marker="D",
        color=PALETTE["B"],
        lw=1.8,
        ls="-.",
        label=r"Kottke Conformal $|T_{-1} - T_{-1}^{\mathrm{RCWA}}|$",
    )

    ref_o2 = err_k_rms[0] * (cpw_lv / float(cpw_lv[0])) ** (-2.0)
    ax.loglog(cpw_lv, ref_o2, color=PALETTE["ink"], ls=":", lw=1.4, label=r"$\mathcal{O}(\Delta x^2)$ slope")

    ax.set_xlabel(r"Resolution [cells per minimum material wavelength $\lambda_{\min}/n_{\max}$]")
    ax.set_ylabel("Diffraction efficiency error vs multi-layer RCWA")
    ax.set_title(r"(a) Staircase vs Conformal Convergence ($\alpha=25^\circ$)")
    ax.legend(loc="lower left", fontsize=7.3, framealpha=0.92)

    # Helper to tile 3 unit cells laterally
    def _tile_3(x_1d, phasor_2d, eps_2d, kx_val):
        xs_l, ph_l, ep_l = [], [], []
        for mt in (-1, 0, 1):
            xs_l.append((x_1d + mt * PERIOD_INCOUPLER_M) * 1e9)
            ph_l.append(phasor_2d * np.exp(-1j * kx_val * (mt * PERIOD_INCOUPLER_M)))
            ep_l.append(eps_2d)
        return np.concatenate(xs_l), np.concatenate(ph_l, axis=0), np.concatenate(ep_l, axis=0)

    # --- Panel (b): Binary Grating (alpha=0) Near-Field Map ---
    ax = axes[1]
    x_b, ph_b, ep_b = _tile_3(
        data_t3["bin_x"], data_t3["bin_phasor"], data_t3["bin_eps"], float(data_t3["bin_kx"])
    )
    z_b = data_t3["bin_z"] * 1e9
    re_b = np.real(ph_b)
    vm_b = max(float(np.percentile(np.abs(re_b), 98.5)), 1e-12)
    pcm_b = ax.pcolormesh(
        x_b, z_b, re_b.T, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-vm_b, vcenter=0.0, vmax=vm_b), shading="auto"
    )
    fig.colorbar(pcm_b, ax=ax, pad=0.02).set_label(r"$\mathrm{Re}(E_y)$ [arb. u.]")
    ax.contour(x_b, z_b, ep_b.T, levels=[2.0, 4.0], colors="black", linewidths=1.1)
    ax.axhline(z_b[14], color=PALETTE["grey"], ls=":", lw=1.1)
    ax.axhline(z_b[-14], color=PALETTE["grey"], ls=":", lw=1.1)
    ax.text(0.0, z_b[6], "Bottom CPML (-z)", ha="center", va="center", fontsize=7.5, bbox=dict(fc="white", alpha=0.8, pad=1.2))
    ax.text(0.0, z_b[-7], "Top CPML (+z)", ha="center", va="center", fontsize=7.5, bbox=dict(fc="white", alpha=0.8, pad=1.2))
    ax.text(
        0.0,
        480.0,
        r"Glass ($n_s=1.74$): Standing Wave ($T_{+1}=T_{-1}$)",
        ha="center",
        fontsize=7.9,
        fontweight="bold",
        bbox=dict(fc="white", ec=PALETTE["B"], alpha=0.88, pad=1.6),
    )
    ax.text(0.0, -450.0, r"Air ($n=1.0$, Normal Incidence $\theta_{\mathrm{in}}=0^\circ$)", ha="center", fontsize=7.9, bbox=dict(fc="white", alpha=0.85, pad=1.5))
    ax.set_xlabel(r"$x$ [nm] ($3\Lambda$ Bloch window)")
    ax.set_ylabel(r"$z$ [nm]")
    ax.set_title(r"(b) Binary ($\alpha=0^\circ$): Symmetric $\pm 1$ Interference")

    # --- Panel (c): Slanted Grating (alpha=25 deg) Near-Field Map ---
    ax = axes[2]
    x_s, ph_s, ep_s = _tile_3(
        data_t3["sl_x"], data_t3["sl_phasor"], data_t3["sl_eps"], float(data_t3["sl_kx"])
    )
    z_s = data_t3["sl_z"] * 1e9
    re_s = np.real(ph_s)
    vm_s = max(float(np.percentile(np.abs(re_s), 98.5)), 1e-12)
    pcm_s = ax.pcolormesh(
        x_s, z_s, re_s.T, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-vm_s, vcenter=0.0, vmax=vm_s), shading="auto"
    )
    fig.colorbar(pcm_s, ax=ax, pad=0.02).set_label(r"$\mathrm{Re}(E_y)$ [arb. u.]")
    ax.contour(x_s, z_s, ep_s.T, levels=[2.0, 4.0], colors="black", linewidths=1.1)
    ax.axhline(z_s[14], color=PALETTE["grey"], ls=":", lw=1.1)
    ax.axhline(z_s[-14], color=PALETTE["grey"], ls=":", lw=1.1)
    ax.text(0.0, z_s[6], "Bottom CPML (-z)", ha="center", va="center", fontsize=7.5, bbox=dict(fc="white", alpha=0.8, pad=1.2))
    ax.text(0.0, z_s[-7], "Top CPML (+z)", ha="center", va="center", fontsize=7.5, bbox=dict(fc="white", alpha=0.8, pad=1.2))
    ax.text(
        0.0,
        480.0,
        r"Glass ($n_s=1.74$): Tilted Guided Wave ($T_{-1} \gg T_{+1}$)",
        ha="center",
        fontsize=7.9,
        fontweight="bold",
        bbox=dict(fc="white", ec=PALETTE["R"], alpha=0.88, pad=1.6),
    )
    ax.text(0.0, -450.0, r"Air ($n=1.0$, Normal Incidence $\theta_{\mathrm{in}}=0^\circ$)", ha="center", fontsize=7.9, bbox=dict(fc="white", alpha=0.85, pad=1.5))
    ax.set_xlabel(r"$x$ [nm] ($3\Lambda$ Bloch window)")
    ax.set_ylabel(r"$z$ [nm]")
    ax.set_title(r"(c) Slanted ($\alpha=25^\circ$): Unidirectional Guided Launch")

    out_path = fig_dir / "05_staircase_vs_conformal_and_nearfields.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path


def plot_fig6_wavefront_gif(data_t3: dict[str, np.ndarray], fig_dir: Path) -> Path:
    """Generate Figure 6: Side-by-side animated GIF (< 5 MB) comparing Binary vs Slanted wavefronts."""
    def _tile_3(x_1d, phasor_2d, eps_2d, kx_val):
        xs_l, ph_l, ep_l = [], [], []
        for mt in (-1, 0, 1):
            xs_l.append((x_1d + mt * PERIOD_INCOUPLER_M) * 1e9)
            ph_l.append(phasor_2d * np.exp(-1j * kx_val * (mt * PERIOD_INCOUPLER_M)))
            ep_l.append(eps_2d)
        return np.concatenate(xs_l), np.concatenate(ph_l, axis=0), np.concatenate(ep_l, axis=0)

    x_b, ph_b, ep_b = _tile_3(
        data_t3["bin_x"], data_t3["bin_phasor"], data_t3["bin_eps"], float(data_t3["bin_kx"])
    )
    z_b = data_t3["bin_z"] * 1e9
    x_s, ph_s, ep_s = _tile_3(
        data_t3["sl_x"], data_t3["sl_phasor"], data_t3["sl_eps"], float(data_t3["sl_kx"])
    )
    z_s = data_t3["sl_z"] * 1e9

    vmax = float(max(np.percentile(np.abs(ph_b), 98.0), np.percentile(np.abs(ph_s), 98.0)))
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.6), constrained_layout=True)
    n_frames = 18
    phases = np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False)

    pcm_b = axes[0].pcolormesh(x_b, z_b, np.real(ph_b).T, cmap="RdBu_r", norm=norm, shading="auto")
    pcm_s = axes[1].pcolormesh(x_s, z_s, np.real(ph_s).T, cmap="RdBu_r", norm=norm, shading="auto")
    cbar = fig.colorbar(pcm_s, ax=axes.tolist(), pad=0.02, shrink=0.92)
    cbar.set_label(r"Electric field $\mathrm{Re}\{E_y(x,z)e^{i\omega t}\}$ [arb. u.]")

    for ax, x_arr, z_arr, ep_arr, r_lbl, top_lbl in [
        (
            axes[0],
            x_b,
            z_b,
            ep_b,
            r"Binary Ridges ($\alpha=0^\circ$, $n=2.35$)",
            r"Glass ($n_s=1.74$): Symmetric $\pm 1$ Standing Wave",
        ),
        (
            axes[1],
            x_s,
            z_s,
            ep_s,
            r"Slanted Ridges ($\alpha=25^\circ$, $n=2.35$)",
            r"Glass ($n_s=1.74$): Unidirectional $m=-1$ Guided Beam",
        ),
    ]:
        ax.contour(x_arr, z_arr, ep_arr.T, levels=[2.0, 4.0], colors="black", linewidths=1.2)
        ax.axhline(z_arr[14], color=PALETTE["grey"], ls=":", lw=1.1)
        ax.axhline(z_arr[-14], color=PALETTE["grey"], ls=":", lw=1.1)
        ax.text(0.0, z_arr[6], "Bottom CPML (-z)", ha="center", va="center", fontsize=7.6, bbox=dict(fc="white", alpha=0.82, pad=1.2))
        ax.text(0.0, z_arr[-7], "Top CPML (+z)", ha="center", va="center", fontsize=7.6, bbox=dict(fc="white", alpha=0.82, pad=1.2))
        ax.text(0.0, 480.0, top_lbl, ha="center", fontsize=8.0, fontweight="bold", bbox=dict(fc="white", ec=PALETTE["B"], alpha=0.9, pad=1.6))
        ax.text(0.0, 160.0, r_lbl, ha="center", fontsize=7.8, bbox=dict(fc="white", ec=PALETTE["R"], alpha=0.88, pad=1.4))
        ax.text(0.0, -450.0, r"Air ($n=1.0$, $\theta_{\mathrm{in}}=0^\circ$, Periodic $\pm x$)", ha="center", fontsize=7.8, bbox=dict(fc="white", alpha=0.85, pad=1.4))
        ax.set_xlabel(r"$x$ [nm] ($3\Lambda$ periodic window)")
        ax.set_ylabel(r"$z$ [nm]")

    t0_obj = axes[0].set_title(r"(a) Binary ($\alpha=0^\circ$, $\omega t = 0.00$ rad)")
    t1_obj = axes[1].set_title(r"(b) Slanted ($\alpha=25^\circ$, $\omega t = 0.00$ rad)")

    def _update(frame_idx: int):
        phi_t = phases[frame_idx]
        pcm_b.set_array(np.real(ph_b * np.exp(1j * phi_t)).T.ravel())
        pcm_s.set_array(np.real(ph_s * np.exp(1j * phi_t)).T.ravel())
        t0_obj.set_text(rf"(a) Binary ($\alpha=0^\circ$, $\omega t = {phi_t:.2f}$ rad)")
        t1_obj.set_text(rf"(b) Slanted ($\alpha=25^\circ$, $\omega t = {phi_t:.2f}$ rad)")
        return pcm_b, pcm_s, t0_obj, t1_obj

    ani = animation.FuncAnimation(fig, _update, frames=n_frames, interval=90, blit=False)
    out_path = fig_dir / "06_slanted_vs_binary_wavefront_movie.gif"
    ani.save(out_path, writer=animation.PillowWriter(fps=11), dpi=105)
    plt.close(fig)
    return out_path

## 3. Workflow Execution & Benchmark Summary

Executing `slanted_grating_coupler.py` runs (or loads with `--plot`) the four tasks,
saves all numerical results in `data/`, and creates Figures 1–6 in `figures/`.

In [ ]:
def main(argv: list[str] | None = None) -> dict[str, object]:
    args = parse_args(argv)
    if args.list_tasks:
        for idx, (slug, desc) in TASKS.items():
            print(f"Task {idx} ({slug}): {desc}")
        return {}

    out_dir: Path = args.out
    fig_dir: Path = args.fig_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    runners = {
        0: lambda: run_task_0_slant_and_profiles(quick=args.quick),
        1: lambda: run_task_1_depth_and_fill_sweeps(quick=args.quick),
        2: lambda: run_task_2_oblique_and_broadband(quick=args.quick, processes=args.processes),
        3: lambda: run_task_3_staircase_vs_conformal(quick=args.quick),
    }

    if args.task is not None:
        t_idx = args.task
        slug, _ = TASKS[t_idx]
        summary, npz_dict = runners[t_idx]()
        np.savez_compressed(out_dir / f"task_{t_idx}_{slug}.npz", **npz_dict)
        (out_dir / f"task_{t_idx}_{slug}.json").write_text(json.dumps(summary, indent=2))
        print(f"[Tutorial 6.5] Saved Task {t_idx} ({slug}) to {out_dir}")
        return summary

    all_summaries: dict[str, object] = {}
    loaded_npz: dict[int, dict[str, np.ndarray]] = {}

    for t_idx, (slug, _) in TASKS.items():
        npz_path = out_dir / f"task_{t_idx}_{slug}.npz"
        json_path = out_dir / f"task_{t_idx}_{slug}.json"
        if args.plot and npz_path.exists() and json_path.exists():
            with np.load(npz_path) as d:
                loaded_npz[t_idx] = {k: d[k] for k in d.files}
            all_summaries[slug] = json.loads(json_path.read_text())
        else:
            summary, npz_dict = runners[t_idx]()
            np.savez_compressed(npz_path, **npz_dict)
            json_path.write_text(json.dumps(summary, indent=2))
            loaded_npz[t_idx] = npz_dict
            all_summaries[slug] = summary

    (out_dir / "summary.json").write_text(json.dumps(all_summaries, indent=2))

    f1 = plot_fig1_schematic_and_meshes(fig_dir)
    f2 = plot_fig2_slant_and_profiles(loaded_npz[0], fig_dir)
    f3 = plot_fig3_depth_and_fill(loaded_npz[1], fig_dir)
    f4 = plot_fig4_oblique_and_broadband(loaded_npz[2], fig_dir)
    f5 = plot_fig5_convergence_and_nearfields(loaded_npz[3], fig_dir)
    f6 = plot_fig6_wavefront_gif(loaded_npz[3], fig_dir)

    print("=== Tutorial 6.5 Summary ===")
    print(json.dumps(all_summaries, indent=2))
    print("Generated figures:")
    for fp in (f1, f2, f3, f4, f5, f6):
        print(f"  - {fp.name} ({fp.stat().st_size / 1024:.1f} kB)")
    return all_summaries


if __name__ == "__main__":
    main()

## 4. Results vs Literature & Resolution Convergence Study

### Comparison against verified literature

| Physical quantity / metric | Literature reference & value | Wakis 2D FIT / Multi-Layer RCWA | Agreement / Notes |
| :--- | :--- | :--- | :--- |
| Binary grating ($\alpha=0^\circ$) $\pm 1$ symmetry at $\theta_{\text{in}}=0^\circ$ | $T_{+1} = T_{-1}$ by mirror symmetry (**Wang & Ma, 2023**, Table 1 OMC) | $|T_{-1} - T_{+1}| = 6.2\times 10^{-17}$ ($T_{-1} = T_{+1} = 4.63\%$, ratio $1.000$) | Exact symmetry verified |
| Geometric Bragg slant angle vs optimal slant angle | $\alpha = 25^\circ$, $\bar{\eta} \approx 80\%$ at $\lambda_0 = 525\;\text{nm}, n_s = 1.74$ (**Wang & Ma, 2023**, Table 1 & Fig. 6, p. 9–11) | $\alpha_{\text{Bragg}} = 24.48^\circ$; full-wave $T_{-1} = 82.3\%$ (Wakis) vs $81.9\%$ (RCWA) at $\alpha = 25^\circ$ | Matches Table 1 slant & $\sim 80\%$ efficiency |
| Slanted grating $\pm 1$ selectivity $T_{-1}/T_{+1}$ | $\sim 8\text{--}16\times$ single-order contrast (**Levola & Laakkonen, 2007**, Fig. 4) | $T_{-1}/T_{+1} = 16.3\times$ at $\alpha = 25^\circ$ ($82.3\%$ vs $5.0\%$) | Strong unidirectional blazing |
| Sawtooth blazed vs Slanted first-order efficiency | Slanted ridges outperform planar sawtooth blaze at high deflection angles (**Levola, 2006**) | Binary $4.6\%$ $\to$ Sawtooth Blazed $43.2\%$ ($12.6\times$) $\to$ Slanted $82.3\%$ ($16.3\times$) | $+77.7\%$ gain over Binary |
| Wakis FIT vs Multi-Layer RCWA order error | Benchmark tolerance $\le 1.5\times 10^{-2}$ (`--quick`) | Max $|\Delta T_{-1}| = 4.4\times 10^{-3}$ ($\alpha$ sweep), $8.9\times 10^{-3}$ ($d$ sweep), $4.8\times 10^{-3}$ (broadband) | Sub-percent agreement |

### Staircase vs Kottke Conformal convergence study (Task 3, $\alpha = 25^\circ$, $\theta_{\text{in}} = 8^\circ$)

Comparing `averaging="staircase"` against `averaging="kottke"` across 4 grid resolutions
($\text{cpw} \in \{16, 20, 26, 34\}$) demonstrates that conformal subpixel smoothing reduces the
diffraction-order RMS error by **$2.5\times\text{--}4.0\times$ at equal cell count**, matching the accuracy
of a staircase mesh that has nearly twice the linear grid resolution.

---

## 5. Limits & Failure Modes

1. **High slant angles ($\alpha > 45^\circ$) and aspect-ratio fabrication limits:**
   When $\alpha$ exceeds $\sim 45^\circ$, the tilted ridge overhangs the adjacent groove by more than
   half a period ($d\tan\alpha > 0.5\Lambda$). In reactive-ion etching (RIE) or nano-imprint lithography
   (**Levola & Laakkonen, 2007**), extreme slant angles suffer from master-stamp demolding collapse and
   incomplete $\text{TiO}_2$ALD/evaporation conformal coating. Numerically, larger $\tan\alpha$ increases
   the horizontal shear per $z$-slice, requiring finer $\Delta z$ to resolve the tilted phase front.
2. **TM polarization ($p$-pol) high-index contrast jump:**
   In TM polarization, the electric field lies in the $(x, z)$ plane and has a large component normal to the
   slanted sidewalls, where $E_n = D_n/\varepsilon$ is discontinuous by $(n_{\text{ridge}}/n_{\text{groove}})^2 = 5.52$.
   Without Kottke's harmonic inverse-permittivity projection $\langle\varepsilon^{-1}\rangle$ along $\hat{\mathbf{n}}$,
   staircase FDTD exhibits severe polarization-dependent frequency shifts.
3. **Over-etching groove depth beyond the first Fabry-Perot half-wave peak:**
   As shown in Figure 3(a), increasing groove depth $d$ beyond $d_{\text{opt}} \approx 350\text{--}415\;\text{nm}$
   drives $\Delta\phi > \pi$, causing power to coupled back into the $0\text{th}$ and $+1\text{st}$ orders.

---

## 6. Exercises
1. **Slant-angle reversal ($\alpha \to -\alpha$):** Change `slant_deg=+25.0` to `-25.0` in `build_slanted_floquet_cell()` and verify that $T_{-1}$ and $T_{+1}$ swap roles exactly at normal incidence.
2. **TiO$_2$ atomic-layer coating ($n_{\text{ridge}} = 2.985$):** Increase `eps_ridge` to $2.985^2$ (**Wang & Ma, 2023**, Table 1 $\text{TiO}_2$ refractive index) and find the reduced groove depth $d_{\text{opt}}$ that maximizes $T_{-1}$.
3. **Two-layer slanted + coating RCWA:** Extend `rcwa_multilayer_1d` to model a polymer master grating ($n=1.55$) conformally over-coated with a $140\;\text{nm}$ $\text{TiO}_2$ film (**Wang & Ma, 2023**, Fig. 5).

---

## 7. Verified References
1. **H. Wang and D. Ma**, *"Surface relief grating near-eye display waveguide design"*, `arXiv:2307.01600` (2023), DOI: [`10.48550/arXiv.2307.01600`](https://doi.org/10.48550/arXiv.2307.01600). (Table 1 & Fig. 6: slanted in-coupler $\alpha = 25^\circ$, fill factor $66\%$, depth $415\;\text{nm}$ at $\lambda = 525\;\text{nm}$ on N-LAF2 substrate $n = 1.74$.)
2. **T. Levola**, *"Diffractive optics for virtual reality displays"*, **J. Soc. Inf. Display** **14**(5), 467–475 (2006), DOI: [`10.1889/1.2206112`](https://doi.org/10.1889/1.2206112).
3. **T. Levola and P. Laakkonen**, *"Replicated slanted gratings with a high refractive index coating for efficient compact coverage of the eye-box"*, **Opt. Express** **15**(5), 2067–2074 (2007), DOI: [`10.1364/OE.15.002067`](https://doi.org/10.1364/OE.15.002067). ($\lambda = 532\;\text{nm}, \Lambda = 405\;\text{nm}$, slant angle $35^\circ$, depth $330\;\text{nm}$.)
4. **A. Farjadpour, D. Roundy, A. Rodriguez, M. Ibanescu, P. Bermel, J. D. Joannopoulos, S. G. Johnson, and G. W. Burr**, *"Improving accuracy by subpixel smoothing in the finite-difference time domain"*, **Opt. Lett.** **31**(20), 2972–2974 (2006), DOI: [`10.1364/OL.31.002972`](https://doi.org/10.1364/OL.31.002972).
5. **N. Kottke, A. Farjadpour, and S. G. Johnson**, *"Perturbation theory for anisotropic dielectric interfaces, and application to subpixel smoothing of discretized numerical methods"*, **Phys. Rev. E** **77**, 036611 (2008), DOI: [`10.1103/PhysRevE.77.036611`](https://doi.org/10.1103/PhysRevE.77.036611).